# 02 — Age-group analysis

Compare estimated entry ages 18–24 and 35+ during 2015–2018. A is 18–24 and B is
35+. The source records age at final exit or the observation cutoff.


In [1]:
from pathlib import Path
import sys
import pandas as pd

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "data/raw.xlsx").is_file()
             and (p / "src/analysis.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from the extracted project folder.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.pipeline import load_data, preparation_tables, monthly_tables, save_tables

raw, prepared, members = load_data()
OUT = ROOT / "results"

from src.analysis import age_results


Estimate entry age as:

`recorded age + 0.5 − elapsed calendar days / 365.2425`

Elapsed time runs from first registration to the age reference date. The 0.5
uses the midpoint of the recorded whole-year age. Group A is 18 ≤ estimate < 25;
group B is estimate ≥ 35. Lower/upper estimates are checked below.


In [2]:
members[["Age", "StartDate", "AgeReferenceDate", "EstimatedEntryAge"]].head()

,Age,StartDate,AgeReferenceDate,EstimatedEntryAge
0,60,2019-09-03,2019-10-31,60.341201
1,29,2014-08-12,2015-09-14,28.410313
2,23,2017-05-02,2017-06-01,23.417863
3,9,2018-09-05,2019-02-12,9.061935
4,35,2016-04-20,2018-06-07,33.369908


In [3]:
tables = age_results(members)
tables["GroupsAndChecks"]

,Measure,A: entry age 18-24,B: entry age 35+
0,AllSourceHistoryRecords,6492,2884
1,ReportingPeriodCustomers,5810,2471
2,FirstEnrolments2015_2018,4899,2118
3,RecordedDropouts2015_2018,4604,1599
4,OpeningJanuary2015,911,353
5,ClosingDecember2018,1206,872
6,MinimumMonthlyOpeningBase,892,353
7,MaximumMonthlyOpeningBase,1780,932
8,BoundarySensitiveReportingCustomers,409,88
9,SameMonthEntrantExits,11,7


Within each age group and year, calculate:

`monthly registration share = monthly new enrolments / annual new enrolments`

Then average each calendar month's four shares for 2015–2018. Monthly dropout
rates use the same group's opening membership. The seasonal table gives the
four-year means used in Tableau. Values below are displayed as percentages;
the saved files retain the original fraction values.


In [4]:
columns = ["MeanAnnualEnrolmentShareA", "MeanAnnualEnrolmentShareB",
           "MeanDropoutRateA", "MeanDropoutRateB"]
view = tables["SeasonalComparison"].set_index("MonthName")[columns].mul(100)
view.columns = ["Registration A (%)", "Registration B (%)", "Dropout A (%)", "Dropout B (%)"]
view.round(2)

,Registration A (%),Registration B (%),Dropout A (%),Dropout B (%)
MonthName,,,,
January,5.45,8.72,4.73,2.95
February,9.95,7.65,3.55,2.18
March,19.81,9.48,2.99,2.81
April,10.82,7.15,3.68,2.49
May,3.89,6.66,4.74,2.54
June,1.22,4.08,7.04,2.63
July,0.64,4.08,20.31,14.01
August,0.52,4.65,4.98,4.09
September,15.31,21.56,15.59,11.56


Sensitivity checks repeat the comparison using the lower age bound,
upper bound, and customers whose group remains stable across the age interval.
All monthly details, annual totals and paired differences are exported.


In [5]:
tables["Sensitivity"][["Scenario", "AgeGroup", "ReportingPeriodCustomers",
                       "MeanEnrolmentRate", "MeanDropoutRate"]]

,Scenario,AgeGroup,ReportingPeriodCustomers,MeanEnrolmentRate,MeanDropoutRate
0,Midpoint estimate,A: entry age 18-24,5810,0.085176,0.066851
1,Midpoint estimate,B: entry age 35+,2471,0.067502,0.044526
2,Lower-age bound,A: entry age 18-24,5871,0.086660,0.068469
3,Lower-age bound,B: entry age 35+,2383,0.066836,0.043999
4,Upper-age bound,A: entry age 18-24,5679,0.085456,0.066347
5,Upper-age bound,B: entry age 35+,2546,0.067576,0.044786
6,Stable group membership only,A: entry age 18-24,5401,0.086004,0.067250
7,Stable group membership only,B: entry age 35+,2383,0.066836,0.043999


In [6]:
save_tables(OUT / "age_groups", tables)
pd.DataFrame({"Saved table": list(tables)})

,Saved table
0,GroupsAndChecks
1,MonthlyData
2,AnnualSummary
3,SeasonalComparison
4,YearlyRateGaps
5,FocusWindows
6,FocusWindowGaps
7,Sensitivity
8,SensitivitySeasonal
